# 🚀 FIXED: Floor Plan AI Training

## Instructions:
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Wait 3-4 hours

**This version FIXES the dtype error!**

In [ ]:
# STEP 1: Mount Drive and Download Dataset
import os
from google.colab import drive

print("="*70)
print("FLOOR PLAN AI TRAINING")
print("="*70)
print("\n📁 Step 1/8: Google Drive...")

drive.mount('/content/drive')
os.chdir('/content')

print("✅ Mounted!")
print("\n📥 Copying dataset (196MB)...")

!cp /content/drive/MyDrive/floor_plan_training/training_data_demo.zip /content/
!mv training_data_demo.zip training_data.zip

print("✅ Ready! 500 floor plans")

In [ ]:
# STEP 2: Install Libraries
print("\n⚙️  Step 2/8: Installing libraries (~5 min)...\n")

!pip install -q diffusers[torch] transformers accelerate datasets pillow torchvision

print("\n✅ Installed!")

In [ ]:
# STEP 3: Extract Data
import zipfile
import os

print("\n📦 Step 3/8: Extracting...")

with zipfile.ZipFile('training_data.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

# Fix folder name
!mv training_data_demo training_data 2>/dev/null || true

train_count = len([f for f in os.listdir('training_data/train/images') if f.endswith('.png')])
val_count = len([f for f in os.listdir('training_data/val/images') if f.endswith('.png')])

print(f"\n✅ Extracted!")
print(f"   Train: {train_count} | Val: {val_count}")

In [ ]:
# STEP 4: Load AI Model
from diffusers import StableDiffusionPipeline, DDPMScheduler
import torch

print("\n🤖 Step 4/8: Loading AI model (~4GB download)...\n")

# Use float32 for stability (no mixed precision issues)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float32,
    safety_checker=None
)
pipe = pipe.to("cuda")

print(f"\n✅ Model loaded! Device: {pipe.device}")

In [ ]:
# STEP 5: Prepare Dataset
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from torchvision import transforms

print("\n📊 Step 5/8: Preparing dataset...")

class FloorPlanDataset(Dataset):
    def __init__(self, data_dir, split='train', size=512):
        self.data_dir = Path(data_dir) / split
        self.image_paths = sorted(list((self.data_dir / 'images').glob('*.png')))
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        prompt_path = self.data_dir / 'prompts' / f"{image_path.stem}.txt"
        with open(prompt_path, 'r') as f:
            prompt = f.read().strip()
        return {'image': self.transform(image), 'prompt': prompt}

train_dataset = FloorPlanDataset('/content/training_data', split='train')
val_dataset = FloorPlanDataset('/content/training_data', split='val')

print(f"\n✅ Dataset ready!")
print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# STEP 6: Configure Training (NO mixed precision - more stable)
from diffusers.optimization import get_cosine_schedule_with_warmup
import torch.nn.functional as F

print("\n⚙️  Step 6/8: Configuring...")

config = {
    'learning_rate': 1e-5,
    'batch_size': 2,  # Smaller batch for fp32
    'num_epochs': 5,   # Fewer epochs for demo
    'save_every': 50,
    'sample_every': 25,
    'output_dir': 'floor_plan_model'
}

train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=config['learning_rate'])

num_training_steps = len(train_dataloader) * config['num_epochs']
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=50, num_training_steps=num_training_steps
)

print(f"\n✅ Configured!")
print(f"   Epochs: {config['num_epochs']} | Steps: {num_training_steps:,}")
print(f"   Time: ~3-4 hours")

In [ ]:
# STEP 7: TRAIN (3-4 hours)
from tqdm.auto import tqdm
from datetime import datetime
import os

os.makedirs(config['output_dir'], exist_ok=True)
os.makedirs(f"{config['output_dir']}/samples", exist_ok=True)

print("\n" + "="*70)
print("🚀 TRAINING STARTED!")
print("="*70)
print(f"Start: {datetime.now().strftime('%H:%M')}")
print("\nYou can close this tab - training continues!")
print("Check back in 3-4 hours\n" + "="*70 + "\n")

global_step = 0
best_loss = float('inf')

for epoch in range(config['num_epochs']):
    pipe.unet.train()
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
    
    for batch in progress_bar:
        # FIXED: Proper dtype handling
        images = batch['image'].to(device='cuda', dtype=torch.float32)
        
        with torch.no_grad():
            latents = pipe.vae.encode(images).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
        
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, 
                                   (latents.shape[0],), device='cuda').long()
        
        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
        
        encoder_hidden_states = pipe.text_encoder(
            pipe.tokenizer(batch['prompt'], padding='max_length', 
                          max_length=77, truncation=True, 
                          return_tensors='pt').input_ids.to('cuda')
        )[0]
        
        noise_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred, noise)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pipe.unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        global_step += 1
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # Save best
        if global_step % config['save_every'] == 0 and loss.item() < best_loss:
            best_loss = loss.item()
            pipe.save_pretrained(f"{config['output_dir']}/best_model")
            print(f"\n✅ Saved! Loss: {best_loss:.4f}")
        
        # Sample
        if global_step % config['sample_every'] == 0:
            pipe.unet.eval()
            with torch.no_grad():
                img = pipe("Floor plan with 3 bedrooms, 2 bathrooms, kitchen", 
                          num_inference_steps=20).images[0]
                img.save(f"{config['output_dir']}/samples/step_{global_step}.png")
            pipe.unet.train()

print("\n" + "="*70)
print("✅ TRAINING DONE!")
print("="*70)
print(f"End: {datetime.now().strftime('%H:%M')}")
print(f"Best loss: {best_loss:.4f}")

In [ ]:
# STEP 8: Test!
from IPython.display import display

print("\n🎨 Testing AI...\n")

pipe = StableDiffusionPipeline.from_pretrained(
    f"{config['output_dir']}/best_model", torch_dtype=torch.float32
).to("cuda")

for i, prompt in enumerate([
    "Floor plan with 2 bedrooms, 1 bathroom, kitchen",
    "Floor plan with 3 bedrooms, 2 bathrooms",
    "Floor plan with 4 bedrooms, 3 bathrooms, kitchen, living room"
]):
    print(f"{i+1}. {prompt}")
    img = pipe(prompt, num_inference_steps=50).images[0]
    img.save(f"result_{i+1}.png")
    display(img)

print("\n🎉 DONE! Your AI can now generate floor plans!")